In [0]:
%run ../config/config

In [0]:
!pip install jinja2==3.1.6

In [0]:
# Imports estándar
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from jinja2 import Environment, FileSystemLoader
from datetime import datetime
import sys
import time

# Configurar path del proyecto
sys.path.insert(0, str(project_path))

# Imports de módulos reportes
from utils.reporting import HTMLReportGenerator
from utils.reporting import ChartGenerator
from utils.reporting import TableBuilder
from utils.reporting import HTMLFormatter

# Imports de utilidades
from utils.logger import MLOpsLogger
from utils.metadata_serializer import MetadataSerializer

print("Imports completados exitosamente")

In [0]:
# =============================================================================
# CONFIGURACIÓN DE GENERACIÓN DE GRÁFICOS
# =============================================================================

# Flag para controlar la generación de gráficos de distribución
GENERATE_DISTRIBUTION_CHARTS = False

# Flag para controlar la generación de gráficos de drift
GENERATE_DRIFT_CHARTS = False

print(f"Configuración de gráficos:")
print(f"  - Gráficos de distribución: {'ACTIVADOS' if GENERATE_DISTRIBUTION_CHARTS else 'DESACTIVADOS'}")
print(f"  - Gráficos de drift: {'ACTIVADOS' if GENERATE_DRIFT_CHARTS else 'DESACTIVADOS'}")

In [0]:
# Logger para generación de reportes
logger = MLOpsLogger(
    name='06_quality_outputs',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=logs_path,
    is_job_run=True,
    job_context={
        'mes_vta': mes_vta,
        'environment': env,
        'notebook': 'generate_reports_from_metadata'
    }
)

In [0]:
# =============================================================================
# CARGAR METADATOS DESDE JSONs INDIVIDUALES
# =============================================================================
# Los notebooks 02_engine_quality y 03_engine_drift generan archivos
# separados en metadata/. Este notebook los carga individualmente.
# =============================================================================
try:
    logger.info("=" * 60)
    logger.info("CARGANDO METADATOS DESDE JSONs INDIVIDUALES")
    logger.info("=" * 60)

    start_time = time.time()

    metadata_dir = os.path.join(temp_reports_output_path, "metadata/global")

    if not os.path.exists(metadata_dir):
        raise FileNotFoundError(
            f"No se encontró el directorio de metadatos: {metadata_dir}\n"
            f"Asegúrate de ejecutar primero 02_engine_quality y 03_engine_drift"
        )

    # Listar archivos disponibles
    available_files = os.listdir(metadata_dir)
    logger.info(f"Archivos en metadata/: {available_files}")

    # =========================================================================
    # 1) QUALITY METADATA (bundle del notebook 02)
    # =========================================================================
    quality_bundle_path = os.path.join(metadata_dir, "quality_metadata.json")
    # Fallback: intentar con full_metadata.json si existe (compatibilidad)
    if not os.path.exists(quality_bundle_path):
        quality_bundle_path = os.path.join(metadata_dir, "full_metadata.json")

    if not os.path.exists(quality_bundle_path):
        raise FileNotFoundError(
            f"No se encontró quality_metadata.json ni full_metadata.json en {metadata_dir}\n"
            f"Ejecuta primero 02_engine_quality.ipynb"
        )

    logger.info(f"Cargando quality bundle: {quality_bundle_path}")
    metadata_bundle = MetadataSerializer.load_metadata_bundle(quality_bundle_path)

    # Extraer componentes del quality bundle
    metadata_info = metadata_bundle['metadata']
    current_metrics_data = metadata_bundle['current_metrics']
    evolutive_metrics_data = metadata_bundle['evolutive_metrics']
    additional_metadata = metadata_bundle.get('additional', {})
    distribution_data_path = metadata_bundle['paths']['distribution_data']

    logger.info(f" ✓ Quality bundle cargado")
    logger.info(f"   Mes: {metadata_info['mes_vta']} | Registros: {metadata_info['total_records']:,} | Vars: {metadata_info['variables_count']}")

    # Convertir current_metrics a DataFrames
    current_metrics = {
        'summary_metrics': pd.DataFrame(current_metrics_data.get('summary_metrics', [])),
        'variable_metrics': pd.DataFrame(current_metrics_data.get('variable_metrics', [])),
        'variable_types': current_metrics_data.get('variable_types', {}),
        'kpis': current_metrics_data.get('kpis', {})
    }

    logger.info(f"   Variables en current_metrics: {len(current_metrics['variable_metrics'])}")

    # Reconstruir evolutive_metrics con índice
    evolutive_metrics = {}
    for metric_name, data in evolutive_metrics_data.items():
        df = pd.DataFrame(data)

        # Buscar columna Variable (puede ser 'Variable' o '("Variable", "")')
        variable_col = None
        for col in df.columns:
            col_str = str(col)
            if col_str == 'Variable' or col_str == '("Variable", "")':
                variable_col = col
                break

        if variable_col is not None:
            df = df.set_index(variable_col)
            df.index.name = 'Variable'  # Normalizar nombre del índice
        evolutive_metrics[metric_name] = df

    logger.info(f"   Evolutive metrics: {len(evolutive_metrics)} métricas")

    # Info adicional
    meses_historicos_count = additional_metadata.get('meses_historicos', 0)
    chart_bins = additional_metadata.get('chart_bins', 30)
    rows_per_page = additional_metadata.get('rows_per_page', 25)

    # =========================================================================
    # 2) DRIFT METADATA (del notebook 03)
    # =========================================================================
    drift_results = None
    variable_comparison = None
    semaphore_kpis = None
    semaphore_config_drift = None

    # Intentar cargar drift_metrics.json (generado por serialize_drift_metrics)
    drift_metrics_path = os.path.join(metadata_dir, "drift_metrics.json")
    drift_bundle_path = os.path.join(metadata_dir, "drift_metadata.json")

    if os.path.exists(drift_metrics_path):
        logger.info(f"Cargando drift metrics: {drift_metrics_path}")
        with open(drift_metrics_path, 'r', encoding='utf-8') as f:
            drift_data = json.load(f)

        # Extraer componentes del drift
        drift_key = 'drift_results' if 'drift_results' in drift_data else 'drift_metrics'
        if drift_data.get(drift_key):
            drift_results = pd.DataFrame(drift_data[drift_key])
            logger.info(f"  ✓ Drift results: {len(drift_results)} variables")

        # Variable comparison
        var_comp_data = drift_data.get('variable_comparison', [])
        if isinstance(var_comp_data, dict) and var_comp_data:
            try:
                variable_comparison = pd.DataFrame(var_comp_data)
            except ValueError:
                variable_comparison = var_comp_data
        elif isinstance(var_comp_data, list) and var_comp_data:
            variable_comparison = pd.DataFrame(var_comp_data)
        else:
            variable_comparison = None

        semaphore_kpis = drift_data.get('semaphore_kpis', {})
        semaphore_config_drift = drift_data.get('semaphore_config', {})

    elif os.path.exists(drift_bundle_path):
        # Fallback: cargar desde drift_metadata.json (el bundle manual)
        logger.info(f"Cargando drift bundle: {drift_bundle_path}")
        with open(drift_bundle_path, 'r', encoding='utf-8') as f:
            drift_bundle = json.load(f)

        drift_summary = drift_bundle.get('drift_summary', {})
        semaphore_kpis = drift_summary.get('semaphore_kpis', {})
        logger.info(f" ✓ Drift bundle cargado (resumen solamente, sin detalle por variable)")
        logger.warning(f"   ⚠️ drift_metrics.json no encontrado - el reporte de drift tendrá info limitada")
    else:
        logger.info("  🚫 No hay archivos de drift disponibles")

    # =========================================================================
    # 3) EVOLUTIVE SEMAPHORES (opcionales, del notebook 02)
    # =========================================================================
    evolutive_semaphore_data = None
    evolutive_kpis_from_file = None

    evol_sem_path = os.path.join(metadata_dir, "evolutive_semaphores.json")
    if os.path.exists(evol_sem_path):
        logger.info(f"Cargando evolutive semaphores: {evol_sem_path}")
        with open(evol_sem_path, 'r', encoding='utf-8') as f:
            evol_sem_data = json.load(f)

        if evol_sem_data.get('evolutive_semaphores'):
            evolutive_semaphore_data = pd.DataFrame(evol_sem_data['evolutive_semaphores'])
            evolutive_kpis_from_file = evol_sem_data.get('evolutive_kpis', {})
            logger.info(f"  ✓ Evolutive semaphores cargados")

    logger.info(f"\n✓ Todos los metadatos cargados exitosamente")

except Exception as e:
    logger.log_exception(
        operation='load_metadata',
        exception=e,
        should_raise=True
    )

In [0]:
# =============================================================================
# GENERAR GRÁFICOS DE DISTRIBUCIÓN DESDE METADATOS
# =============================================================================
try:
    if GENERATE_DISTRIBUTION_CHARTS:
        logger.info("=" * 60)
        logger.info("GENERANDO GRÁFICOS DE DISTRIBUCIÓN")
        logger.info("=" * 60)

        charts_start = time.time()

        logger.info(f"Cargando datos de distribución desde: {distribution_data_path}")

        with open(distribution_data_path, 'r', encoding='utf-8') as f:
            distribution_data = json.load(f)

        logger.info(f"✓ Datos de distribución cargados: {len(distribution_data)} variables")

        def fig_to_base64(fig):
            import base64
            from io import BytesIO
            buf = BytesIO()
            fig.savefig(buf, format='png', bbox_inches='tight', dpi=100)
            buf.seek(0)
            return base64.b64encode(buf.getvalue()).decode('utf-8')

        distribution_charts = {}

        for var_name, var_data in distribution_data.items():
            try:
                bin_edges = var_data['bin_edges']
                counts = var_data['counts']
                stats = var_data['stats']

                fig, ax = plt.subplots(figsize=(10, 6))

                bin_centers = [(bin_edges[i] + bin_edges[i+1]) / 2 for i in range(len(bin_edges)-1)]
                bin_widths = [bin_edges[i+1] - bin_edges[i] for i in range(len(bin_edges)-1)]

                ax.bar(bin_centers, counts, width=bin_widths, edgecolor='k', alpha=0.7)
                ax.set_title(f"Distribución de {var_name}")
                ax.set_xlabel(var_name)
                ax.set_ylabel("Frecuencia")

                stats_text = f"Mean: {stats['mean']:.2f}\nStd: {stats['std']:.2f}\nCount: {stats['count']:,}"
                ax.text(0.02, 0.98, stats_text, transform=ax.transAxes,
                        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

                plt.tight_layout()

                b64_string = f"data:image/png;base64,{fig_to_base64(fig)}"
                distribution_charts[f"Histograma_{var_name}"] = b64_string
                plt.close(fig)

            except Exception as col_error:
                logger.warning(f"  Error al generar gráfico para {var_name}: {str(col_error)}")
                continue

        charts_duration = time.time() - charts_start
        logger.info(f"✓ {len(distribution_charts)} gráficos generados en {charts_duration:.2f}s")
    else:
        logger.info("📊 Gráficos de distribución desactivados (GENERATE_DISTRIBUTION_CHARTS=False)")
        distribution_charts = {}
        charts_duration = 0

except Exception as e:
    logger.log_exception(
        operation='generate_charts',
        exception=e,
        should_raise=True
    )

In [0]:
# =============================================================================
# SINCRONIZACIÓN HACIA ADLS2 (SOLO DEV/PROD)
# =============================================================================
try:
    if SYNC_TO_ADLS and ADLS_ARTIFACTS_PATH:
        logger.log_stage_start('adls2_sync', environment=env)
        import time
        start_time = time.time()

        from utils.adls_sync import sync_artifacts_to_adls

        is_inputs = 'inputs' in logger.name
        local_reports_path = temp_reports_input_path if is_inputs else temp_reports_output_path

        # 1. Sincronizar Reportes y Metadatos
        adls_reports_dir = f"{ADLS_ARTIFACTS_PATH}/reports/{'inputs' if is_inputs else 'outputs'}/{mes_vta}"
        sync_artifacts_to_adls(
            dbutils=dbutils,
            local_dir=local_reports_path,
            adls_dir=adls_reports_dir,
            logger=logger
        )

        # 2. Sincronizar Logs del Pipeline Completo
        adls_logs_dir = f"{ADLS_ARTIFACTS_PATH}/logs/{mes_vta}"
        sync_artifacts_to_adls(
            dbutils=dbutils,
            local_dir=logs_path,
            adls_dir=adls_logs_dir,
            logger=logger
        )

        logger.log_stage_end('adls2_sync', duration=time.time() - start_time)
    else:
        logger.info("Entorno local: Sincronización a ADLS2 omitida.")

except Exception as e:
    logger.log_exception(operation='adls2_sync', exception=e, should_raise=False)

In [0]:
# =============================================================================
# GENERACIÓN DE REPORTE UNIFICADO – Quality + Drift (Inputs + Outputs)
# =============================================================================

import os
import json
from pathlib import Path

# — Rutas ——————————————————————————————————————————————————————————————
input_metadata_base = Path(temp_reports_input_path) / "metadata"
output_metadata_base = Path(temp_reports_output_path) / "metadata"

reports_path = temp_reports_output_path
unified_path = os.path.join(reports_path, f"reporte_unificado_{env}_{mes_vta}.html")

print(f"Detectando segmentos en inputs: {input_metadata_base}")
print(f"Detectando segmentos en outputs: {output_metadata_base}")

# — Detectar segmentos ————————————————————————————————————————————————
def get_segments(base_dir):
    base = Path(base_dir)
    if not base.exists():
        return []
    return sorted({d.name for d in base.iterdir() if d.is_dir()})

input_segments = get_segments(input_metadata_base)
output_segments = get_segments(output_metadata_base)
segments = sorted(set(input_segments) | set(output_segments)) or ["global"]

print(f"Segmentos inputs: {input_segments}")
print(f"Segmentos outputs: {output_segments}")
print(f"Segmentos a procesar: {segments}")

# — Helpers ———————————————————————————————————————————————————————————
import pandas as pd

def load_bundle(bundle_path):
    p = Path(bundle_path)
    if not p.exists():
        return None
    with open(p, "r", encoding="utf-8") as fh:
        return json.load(fh)

def bundle_to_current_metrics(bundle):
    if bundle is None:
        return {}
    try:
        curr = bundle.get("current_metrics", {})
        vm = curr.get("variable_metrics") or curr.get("metrics") or curr.get("variables")
        sm = curr.get("summary_metrics") or curr.get("summary") or curr.get("semaphore")
        vt = curr.get("variable_types") or curr.get("types") or {}
        result = {}
        if vm is not None:
            result["variable_metrics"] = pd.DataFrame(vm) if isinstance(vm, (list, dict)) else vm
        if sm is not None:
            result["summary_metrics"] = pd.DataFrame(sm) if isinstance(sm, (list, dict)) else sm
        if vt:
            result["variable_types"] = vt
        return result
    except Exception as ex:
        print(f"! bundle_to_current_metrics: {ex}")
        return {}

def bundle_to_evolutive(bundle):
    if bundle is None:
        return {}
    evol = bundle.get("evolutive_metrics") or {}
    result = {}
    for metric_name, data in evol.items():
        try:
            df = pd.DataFrame(data) if isinstance(data, (list, dict)) else data
            if "Variable" in df.columns:
                df = df.set_index("Variable")
            result[metric_name] = df
        except Exception as ex:
            print(f"! evolutive[{metric_name}]: {ex}")
    return result

def bundle_to_drift(bundle):
    if bundle is None:
        return None
    drift_data = bundle.get("drift_results") or bundle.get("drift")
    if drift_data is None:
        return None
    try:
        if isinstance(drift_data, dict) and "drift_metrics" in drift_data:
            drift_data = drift_data["drift_metrics"]
        return pd.DataFrame(drift_data) if isinstance(drift_data, (list, dict)) else drift_data
    except Exception as ex:
        print(f"! bundle_to_drift: {ex}")
        return None

# — Score buckets desde distribution_data de outputs ——————————————
PD_CUTS = [0.01, 0.03, 0.07, 0.15, 0.30]

def build_buckets_pd(dist, cuts):
    """Construye buckets con cortes PD a partir de distribution_data de una variable."""
    stats = dist.get("stats", {})
    total = int(stats.get("count", 0) or 0)
    null_count = int(stats.get("null_count", 0) or 0)
    grand_total = total + null_count
    bin_edges = dist.get("bin_edges", [])
    counts = dist.get("counts", [])

    cut_points = [0.0] + cuts + [float("inf")]
    buckets = []

    if null_count > 0:
        pct = null_count / grand_total * 100
        buckets.append({"label": "C0: NULL", "count": null_count, "pct": round(pct, 4)})

    for ci in range(len(cut_points) - 1):
        lo_cut, hi_cut = cut_points[ci], cut_points[ci + 1]
        cnt = 0
        for j, c in enumerate(counts):
            if j >= len(bin_edges) - 1:
                continue
            b_lo, b_hi = bin_edges[j], bin_edges[j + 1]
            if b_lo >= hi_cut or b_hi <= lo_cut:
                continue
            cnt += c
        pct = cnt / grand_total * 100 if grand_total > 0 else 0
        hi_label = f"{hi_cut:.2f}" if hi_cut != float("inf") else "∞"
        buckets.append({
            "label": f"C{ci+1}: [{lo_cut:.2f}, {hi_label})",
            "count": int(cnt),
            "pct": round(pct, 4)
        })

    return {"total": grand_total, "buckets": buckets, "stats": stats}

def build_score_buckets_for_seg(dist_data):
    """Extrae score buckets de distribution_data de un segmento."""
    if not dist_data:
        return {}
    seg_buckets = {}
    # Variables SC_* / SCORE en el nombre
    for v in dist_data:
        if v.upper().startswith("SC_") or "SCORE" in v.upper():
            dist = dist_data[v]
            bin_edges = dist.get("bin_edges", [])
            counts = dist.get("counts", [])
            stats = dist.get("stats", {})
            total = int(stats.get("count", sum(counts)) or sum(counts))
            bkts = []
            for i, cnt in enumerate(counts):
                if i < len(bin_edges) - 1:
                    lo, hi = bin_edges[i], bin_edges[i+1]
                    label = f"B{i+1}: [{lo:.4f}, {hi:.4f})"
                else:
                    label = f"B{i+1}"
                pct = (cnt / total * 100) if total > 0 else 0
                bkts.append({"label": label, "count": int(cnt), "pct": round(pct, 4)})
            seg_buckets[v] = {"total": total, "buckets": bkts, "stats": stats}
    # numpd con cortes PD
    if "numpd" in dist_data:
        seg_buckets["numpd (PD Score)"] = build_buckets_pd(dist_data["numpd"], PD_CUTS)
    return seg_buckets

# — Construir dicts por segmento ————————————————————————————————
inputs_current_by_seg = {}
inputs_evolutive_by_seg = {}
inputs_drift_by_seg = {}
inputs_distribution_by_seg = {}
inputs_table_name_by_seg = {}

outputs_current_by_seg = {}
outputs_evolutive_by_seg = {}
outputs_drift_by_seg = {}
outputs_distribution_by_seg = {}
outputs_table_name_by_seg = {}

score_buckets_by_seg = {}

for seg in segments:
    print(f"\nProcesando segmento: [{seg}]")

    # Inputs
    in_q = load_bundle(input_metadata_base / seg / "quality_metadata.json")
    in_dr = load_bundle(input_metadata_base / seg / "drift_metadata.json")
    in_di = load_bundle(input_metadata_base / seg / "distribution_data.json")

    inputs_current_by_seg[seg] = bundle_to_current_metrics(in_q)
    inputs_evolutive_by_seg[seg] = bundle_to_evolutive(in_q)
    inputs_drift_by_seg[seg] = bundle_to_drift(in_dr)
    inputs_distribution_by_seg[seg] = in_di or {}
    inputs_table_name_by_seg[seg] = (in_q or {}).get("metadata", {}).get("table_name", "")

    n_iv = len(inputs_current_by_seg[seg].get("variable_metrics", pd.DataFrame()))
    n_id = len(inputs_drift_by_seg[seg]) if inputs_drift_by_seg[seg] is not None else 0
    print(f"  inputs → {n_iv} vars calidad | {n_id} vars drift | {inputs_table_name_by_seg[seg]}")

    # Outputs
    out_q = load_bundle(output_metadata_base / seg / "quality_metadata.json")
    out_dr = load_bundle(output_metadata_base / seg / "drift_metadata.json")
    out_di = load_bundle(output_metadata_base / seg / "distribution_data.json")

    outputs_current_by_seg[seg] = bundle_to_current_metrics(out_q)
    outputs_evolutive_by_seg[seg] = bundle_to_evolutive(out_q)
    outputs_drift_by_seg[seg] = bundle_to_drift(out_dr)
    outputs_distribution_by_seg[seg] = out_di or {}
    outputs_table_name_by_seg[seg] = (out_q or {}).get("metadata", {}).get("table_name", "")

    n_ov = len(outputs_current_by_seg[seg].get("variable_metrics", pd.DataFrame()))
    n_od = len(outputs_drift_by_seg[seg]) if outputs_drift_by_seg[seg] is not None else 0
    print(f"  outputs → {n_ov} vars calidad | {n_od} vars drift | {outputs_table_name_by_seg[seg]}")

    # Score buckets – preferimos score_distribution_by_month.json (multi-periodo)
    score_hist = load_bundle(output_metadata_base / seg / "score_distribution_by_month.json")
    seg_bkts = {}

    if score_hist:
        for raw_var, var_data in score_hist.items():
            by_period = var_data.get("by_period", {})
            if not by_period:
                continue
            sorted_periods = sorted(by_period.keys())
            latest = sorted_periods[-1]
            latest_data = by_period[latest]
            display_name = "numpd (PD Score)" if "numpd" in raw_var.lower() else raw_var
            seg_bkts[display_name] = {
                "total": latest_data.get("total", 0),
                "buckets": latest_data.get("buckets", []),
                "periods": {p: by_period[p]["buckets"] for p in sorted_periods}
            }
        print(f"  score → {list(seg_bkts.keys())} ({len(sorted_periods)} periodos)")
    else:
        seg_bkts = build_score_buckets_for_seg(out_di or {})
        print(f"  score → {list(seg_bkts.keys()) if seg_bkts else 'sin datos'}")

    if seg_bkts:
        score_buckets_by_seg[seg] = seg_bkts

# — Thresholds ———————————————————————————————————————————————————————————
try:
    from utils.config_loader import ConfigLoader
    _cfg = ConfigLoader(quality_config).get_all_input_quality_config()
    amber_thr = _cfg.get("evolutive_thresholds", {}).get("yellow", 5) / 100
    red_thr = _cfg.get("evolutive_thresholds", {}).get("red", 25) / 100
except Exception:
    amber_thr, red_thr = 0.05, 0.20

default_thresholds = {"amber": amber_thr, "red": red_thr}
print(f"\nThresholds: amber={amber_thr:.2f}, red={red_thr:.2f}")

# — Generar reporte unificado ———————————————————————————————————————————
print("\n" + "=" * 60)
print("Generando reporte unificado...")
print("=" * 60)

os.makedirs(reports_path, exist_ok=True)

import importlib
import utils.reporting.html_generator as _hrg_mod
importlib.reload(_hrg_mod)
from utils.reporting.html_generator import HTMLReportGenerator

generator = HTMLReportGenerator()
generator.generate_unified_report(
    path_out=unified_path,
    model_id=use_case,
    period=str(mes_vta),
    inputs_by_segment=inputs_current_by_seg,
    outputs_by_segment=outputs_current_by_seg,
    inputs_evolutive_by_segment=inputs_evolutive_by_seg,
    outputs_evolutive_by_segment=outputs_evolutive_by_seg,
    inputs_drift_by_segment=inputs_drift_by_seg,
    outputs_drift_by_segment=outputs_drift_by_seg,
    inputs_distribution_by_segment=inputs_distribution_by_seg,
    outputs_distribution_by_segment=outputs_distribution_by_seg,
    inputs_table_name_by_segment=inputs_table_name_by_seg,
    outputs_table_name_by_segment=outputs_table_name_by_seg,
    score_buckets_by_segment=score_buckets_by_seg if score_buckets_by_seg else None,
    default_thresholds=default_thresholds,
)

print(f"\n✅ Reporte unificado generado:")
print(f"  {unified_path}")

# TaskValues
try:
    dbutils.jobs.taskValues.set(key="reports_generated", value=json.dumps([unified_path]))
except Exception:
    pass